# Chunked-Prefill

## Chunked—Prefill 示例

解码过程特性：

1. Prefill: Compute-bound
2. Decoding: Memory-bound，我们熟知的计算 attention 前要拉取大量的 KVCache，存在显著的 memory 访问的开销。

另外从投影分析：

在 Decoding 时，输入为 next_token, 此时需要拉取一个Attn里的 Wq、Wk、Wv 做投影，或者 FFN 里的 W 权重，计算形式为

`x(1xd) @ Wq(dxd)`

可见我们频繁拉取大的权重矩阵到 SRAM 中，只是做简单的运算，这种访存开销是不经济的。而 Prefill 则是产生了充分的计算的，如：

```
X(Lxd) @ Wq(dxd)
```

此时如果我们定义两个请求：其输入为 
```
req1 (prefill stage): X_req1 [1000xd]
req2 (decoding stage): x_req2 [1xd]
```

我们进行拼接为
```
X_cp = torch.cat( [X_req1, x_req2], dim=0)
Q_cp = X_cp @ Wq
Q_req1, q_req2 <- split(Q_cp)
```

这种处理技巧我们称为 Chunked—Prefill。 为什么不叫 Chunked-Decoding?

1. Prefill 在投影计算中是矩阵乘高效的
2. Decoding 搭了 Prefill 的便车
3. Chunked 的定义，我理解是 `X_cp` 对应有多块（chunked）数据来源

可以理解为通过融合 PD 阶段共性计算，减少 memory-visited 开销，从而提速。

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from typing import Dict, List, Set, Tuple, Optional, Any

torch.manual_seed(42)

In [12]:
d = 2048

# Prefill 和 Decoding 各一个请求
L_p = 1024
L_d = 1 

X_p = torch.randn(L_p, d)
X_d = torch.randn(L_d, d)

W = torch.randn(d,d)

# combine
X_pd = torch.cat( (X_p, X_d), dim = 0)

Y_pd = X_pd @ W

Yp = Y_pd[:L_p, :]
Yd = Y_pd[L_p:, :]

In [13]:
Wq = nn.Linear(d, d)
    
def ChunkPrefillLinearForward(W, XP, XD):
    with torch.no_grad():
        L_p, d = XP.shape
        XPD = torch.cat((XP, XD), dim = 0)
        YPD = W(XPD)
        return YPD[:L_p, :], YPD[L_p:, :]

Yp, Yd = ChunkPrefillLinearForward(Wq, X_p, X_d)
print(Yp.shape)
print(Yd.shape)

torch.Size([1024, 2048])
torch.Size([1, 2048])


In [14]:
### 通用实现

# config 
d = 2048
bsz_p = 3 # prefill请求数量
bsz_d = 20 # decoding请求数量
seq_p = 1024
seq_d = 1 

# data
Wq = nn.Linear(d, d)

def ChunkPrefillLinearForward(W, XP, XD):
    """
    更通用的 Chunk Prefill, 可处理 PD batch size 不同的情况
    """
    
    BP, LP, D = XP.shape
    BD, LD, D = XD.shape
    
    with torch.no_grad():
        XP = XP.reshape(BP*LP, D)
        XD = XD.reshape(BD*LD, D)
        XPD = torch.cat((XP, XD), dim = 0)
        YPD = W(XPD)

        YP = YPD[:BP*LP].reshape(BP, LP, D)
        YD = YPD[BP*LP:].reshape(BD, LD, D)

        
        return YP, YD

X_p = torch.randn(bsz_p, seq_p, d) 
X_d = torch.randn(bsz_d, seq_d, d)

Yp, Yd = ChunkPrefillLinearForward(Wq, X_p, X_d)
print(Yp.shape)
print(Yd.shape)

torch.Size([3, 1024, 2048])
torch.Size([20, 1, 2048])


## Chunked Prefill 对推理系统设计

1. 找到各模块共性计算，进行融合 PD
2. 上述例子描述了 proj 类算子，我们需要进一步分析，注意力计算是否有类似的PD计算共性。

所幸 PD 的注意力遵循：

1. Decoding `单 q 多 KV`
2. Prefill 虽然注意力是 `多 q 多 KV` 计算的， 但其子任务仍为 `单 q 多 KV`

为了节省存储开销，原本所遵循的 kernel 区分了

```
forward_prefill(), page_attention_prefill_kernel()
forward_decoding(), page_attention_decoding_kernel()
```

目标要实现一种不区分 PD 的 通用kernel

```
forward_chunk_prefill(), page_attention_kenrl()
```

恭喜你，发明了 vLLM-V1 版本的 engine step， 计算迭代时间部是不分 PD 任务的。

vLLM-V1 的另外一个特性，乃至是 Inference 系统，会设计成 PD 分离架构，

1. Prefill节点：采用 Chunk-Prefilled 或 standard-Prefilled
2. Decoding节点：对应硬件采用更高速的通信接口，提高访存效率。相应的 Decoding 节点对计算的要求是低于 prefill 节点的。常对 Decoding 节点上更大的 batch size，提高 batch-decoding 效率。

## vLLM-V1 实现逻辑

1. PD 注意力共性计算可以单独优化 Kernel 和 step 流程
2. Chunk-Prefill 仅是一种高效的投影矩阵乘算法

我们将按照以下顺序更新 vLLM-V1

1. PD 混合的 PageAttention kernel，
2. 结合 Chunk-Prefill 实现完整工程
3. 实现 PD 分离